# 02 — Ensembles

## Objetivo

Como Bagging e Boosting se comportam no mesmo problema?

Random Forest representa Bagging; Gradient Boosting é o boosting principal;
XGBoost aparece uma única vez como comparação avançada, sem tuning.

In [1]:
from pathlib import Path
import sys

ponto_atual = Path.cwd().resolve()
RAIZ = next(
    caminho for caminho in (ponto_atual, *ponto_atual.parents)
    if (caminho / "data" / "raw" / "UCI_Credit_Card.csv").exists()
)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


import time
import pandas as pd

from src.auxiliares import (
    avaliar_probabilidades,
    carregar_base_preparada,
    criar_modelo_floresta,
    criar_modelo_gradiente,
    criar_modelo_xgboost,
    separar_dados,
)
from src.visual_utils import grafico_comparacao_modelos

dados = carregar_base_preparada(RAIZ)
X_treino, X_validacao, X_teste, y_treino, y_validacao, y_teste = separar_dados(dados)

## Bagging: qual é a referência?

In [2]:
modelo_floresta = criar_modelo_floresta()
inicio = time.perf_counter()
modelo_floresta.fit(X_treino, y_treino)
tempo_floresta = time.perf_counter() - inicio
probabilidades_floresta = modelo_floresta.predict_proba(X_validacao)[:, 1]

## Boosting: o Gradient Boosting avança?

In [3]:
modelo_gradiente = criar_modelo_gradiente()
inicio = time.perf_counter()
modelo_gradiente.fit(X_treino, y_treino)
tempo_gradiente = time.perf_counter() - inicio
probabilidades_gradiente = modelo_gradiente.predict_proba(X_validacao)[:, 1]

## Quanto o XGBoost acrescenta sem tuning?

In [4]:
modelo_xgboost = criar_modelo_xgboost()
inicio = time.perf_counter()
modelo_xgboost.fit(X_treino, y_treino)
tempo_xgboost = time.perf_counter() - inicio
probabilidades_xgboost = modelo_xgboost.predict_proba(X_validacao)[:, 1]

## A diferença muda a escolha pedagógica?

In [5]:
resultados = pd.DataFrame([
    avaliar_probabilidades("Random Forest", y_validacao, probabilidades_floresta, tempo_treino=tempo_floresta),
    avaliar_probabilidades("Gradient Boosting", y_validacao, probabilidades_gradiente, tempo_treino=tempo_gradiente),
    avaliar_probabilidades("XGBoost", y_validacao, probabilidades_xgboost, tempo_treino=tempo_xgboost),
]).sort_values("pr_auc", ascending=False)

resultados

,modelo,limiar,precision,recall,f1,pr_auc,vn,fp,fn,vp,tempo_treino_s
2,XGBoost,0.5,0.684593,0.354936,0.467494,0.547636,4456,217,856,471,0.178858
1,Gradient Boosting,0.5,0.693314,0.359457,0.473449,0.542580,4462,211,850,477,11.672781
0,Random Forest,0.5,0.654667,0.370008,0.472797,0.529364,4414,259,836,491,1.584769


In [6]:
fig = grafico_comparacao_modelos(resultados, "Bagging e Boosting na validação")
fig.show()

## Resultado

O Gradient Boosting fica muito próximo do XGBoost e mantém a implementação
principal dentro do scikit-learn. O ganho pequeno do XGBoost não muda o foco do
curso.